# Polymarket data exploration

Scratchpad for poking at Polymarket's public APIs. Nothing here is production code.

| API | Base URL | What it gives us |
|---|---|---|
| Gamma | `https://gamma-api.polymarket.com` | events, markets, outcomes, headline prices, volume |
| CLOB | `https://clob.polymarket.com` | order books, price history |
| Data v2 | `https://data-api.polymarket.com/v2` | **trades tied to wallets** (the main input for skill scoring) |

Note: Data API v1 has been retired. v2 filters trades by `condition` (not `market`) and paginates with a cursor.

In [ ]:
import httpx
import pandas as pd
import matplotlib.pyplot as plt
import json

GAMMA = "https://gamma-api.polymarket.com"
CLOB = "https://clob.polymarket.com"
DATA = "https://data-api.polymarket.com/v2"

http = httpx.Client(timeout=15, headers={"User-Agent": "HackTheBruh/0.1"})
pd.set_option("display.max_columns", 50)

## 1. Ping each API

In [ ]:
for name, url in {"gamma": f"{GAMMA}/markets?limit=1", "clob": f"{CLOB}/ok", "data": f"{DATA}/trades?limit=1"}.items():
    r = http.get(url)
    print(f"{name:6} {r.status_code}  {r.elapsed.total_seconds()*1000:.0f} ms")

## 2. Top active markets by volume

In [ ]:
raw = http.get(f"{GAMMA}/markets", params={
    "limit": 50, "active": "true", "closed": "false", "order": "volumeNum", "ascending": "false",
}).json()

# Gamma encodes some list fields as JSON strings
for m in raw:
    for k in ("outcomes", "outcomePrices", "clobTokenIds"):
        if isinstance(m.get(k), str):
            m[k] = json.loads(m[k])

markets = pd.DataFrame(raw)
markets["volume"] = markets["volume"].astype(float)
markets["liquidity"] = pd.to_numeric(markets["liquidity"], errors="coerce")
markets["yes_price"] = markets["outcomePrices"].map(lambda p: float(p[0]) if p else None)
markets[["question", "slug", "conditionId", "yes_price", "volume", "liquidity", "endDate"]].head(20)

## 3. Pick a market and pull its trades (cursor pagination)

In [ ]:
market = markets.iloc[0]
print(market["question"], "|", market["conditionId"])

def fetch_trades(condition_id, max_pages=10, page_size=500):
    rows, cursor = [], None
    for _ in range(max_pages):
        params = {"condition": condition_id, "limit": page_size}
        if cursor:
            params["cursor"] = cursor
        body = http.get(f"{DATA}/trades", params=params).json()
        rows.extend(body["data"])
        page = body.get("pagination", {})
        if not page.get("has_more"):
            break
        cursor = page.get("next_cursor")
    return pd.DataFrame(rows)

trades = fetch_trades(market["conditionId"])
trades["ts"] = pd.to_datetime(trades["timestamp"], unit="s")
trades["notional"] = trades["size"] * trades["price"]
print(len(trades), "trades from", trades["ts"].min(), "to", trades["ts"].max())
trades[["ts", "proxy_wallet", "side", "outcome", "price", "size", "notional"]].head()

## 4. Who is trading? Concentration of volume across wallets

First look at the "crowd behind the number": how concentrated is volume among wallets?

In [ ]:
by_wallet = (trades.groupby("proxy_wallet")
             .agg(n_trades=("notional", "size"), notional=("notional", "sum"),
                  n_buys=("side", lambda s: (s == "BUY").sum()),
                  n_sells=("side", lambda s: (s == "SELL").sum()))
             .sort_values("notional", ascending=False))

share = by_wallet["notional"].cumsum() / by_wallet["notional"].sum()
print(f"{len(by_wallet)} wallets")
for pct in (0.01, 0.05, 0.10):
    k = max(1, int(len(by_wallet) * pct))
    print(f"top {pct:.0%} of wallets ({k}) -> {share.iloc[k-1]:.1%} of notional")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(1, len(share) + 1), share.values)
ax.set_xscale("log"); ax.set_xlabel("wallets (ranked by notional)"); ax.set_ylabel("cumulative share")
ax.set_title("Volume concentration"); ax.grid(alpha=0.3)
by_wallet.head(10)

## 5. Rough wash-trading smell test

A wallet that buys and sells almost equal amounts with little net position is suspicious.
This is only a heuristic to start from, not a detector.

In [ ]:
signed = trades.assign(signed=lambda d: d["size"].where(d["side"] == "BUY", -d["size"]))
flow = signed.groupby(["proxy_wallet", "outcome"]).agg(gross=("size", "sum"), net=("signed", "sum"))
flow["churn_ratio"] = 1 - flow["net"].abs() / flow["gross"]  # 1 = perfectly offsetting
flow.query("gross > 1000").sort_values("churn_ratio", ascending=False).head(15)

## 6. Trade price over time

In [ ]:
yes = trades[trades["outcome_index"] == 0].sort_values("ts")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.scatter(yes["ts"], yes["price"], s=(yes["notional"] / yes["notional"].max() * 200).clip(2), alpha=0.4)
ax.set_ylabel(f"price of '{yes['outcome'].iloc[0]}'" if len(yes) else "price")
ax.set_title(market["question"]); ax.grid(alpha=0.3)

## 7. One wallet's history across all markets (the raw input for a track record)

In [ ]:
wallet = by_wallet.index[0]
w = http.get(f"{DATA}/trades", params={"user": wallet, "limit": 500}).json()["data"]
wdf = pd.DataFrame(w)
print(wallet, "->", len(wdf), "recent trades across", wdf["condition_id"].nunique(), "markets")
wdf[["title", "side", "outcome", "price", "size"]].head(10)

## Next ideas
- Resolved markets: Gamma `closed=true` gives the winning outcome, which lets us score each wallet (Brier / log loss / PnL vs. entry price).
- Separate skill from luck: per-wallet calibration with a shrinkage prior, or a permutation test against random entry.
- Disagreement among skilled wallets: dispersion of skill-weighted positions vs. the size of the move at resolution.
- CLOB `/prices-history?market=<token_id>&interval=max` for a clean price series.